# Collect MICAD Evaluation Results

Aggregates response files for the real-reference evaluation graphs (`cancer`, `earthquake`, `asia`, `sachs`) across OpenAI/open-weight models, semantic cue conditions, data formats, and data budgets.

The notebook is intentionally conservative:
- uses `.per_row.csv` metrics when available;
- falls back to `.summary.json` when per-row metrics are missing;
- normalizes historical `summary_joint` labels to `summary` for paper tables;
- preserves raw filenames/paths for provenance.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import math
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

REPO = Path.cwd()
if not (REPO / "experiments").exists() and (REPO.parent / "experiments").exists():
    REPO = REPO.parent

GRAPHS = ["cancer", "earthquake", "asia", "sachs"]
RESULT_ROOTS = [REPO / "scripts" / "responses", REPO / "experiments" / "responses"]
OUT_DIR = REPO / "experiments" / "out" / "micad_eval_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MODELS = {
    # OpenAI aliases used in local files
    "gpt-5-mini": ["gpt-5-mini", "gpt-4o-mini"],
    "gpt-5.2-pro": ["gpt-5.2-pro", "gpt-4o"],

    # User-specified open-weight sweep
    "Qwen/Qwen3-4B-Thinking-2507": ["Qwen3-4B-Thinking-2507", "Qwen/Qwen3-4B-Thinking-2507"],
    "meta-llama/Meta-Llama-3.1-8B": ["Meta-Llama-3.1-8B", "meta-llama/Meta-Llama-3.1-8B"],
    "meta-llama/Meta-Llama-3.1-8B-Instruct": ["Meta-Llama-3.1-8B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct"],
    "Qwen/Qwen2.5-14B-Instruct-1M": ["Qwen2.5-14B-Instruct-1M", "Qwen/Qwen2.5-14B-Instruct-1M"],
    "Qwen/Qwen2.5-7B-Instruct-1M": ["Qwen2.5-7B-Instruct-1M", "Qwen/Qwen2.5-7B-Instruct-1M"],
    "meta-llama/Llama-3.1-70B-Instruct": ["Llama-3.1-70B-Instruct", "meta-llama/Llama-3.1-70B-Instruct"],
    "Qwen/Qwen3-30B-A3B-Thinking-2507": ["Qwen3-30B-A3B-Thinking-2507", "Qwen/Qwen3-30B-A3B-Thinking-2507"],
    "Qwen/Qwen2.5-72B-Instruct-AWQ": ["Qwen2.5-72B-Instruct-AWQ", "Qwen/Qwen2.5-72B-Instruct-AWQ"],
}

# Match longest aliases first so 70B does not get confused with 8B, etc.
ALIAS_TO_MODEL = {}
for canonical, aliases in TARGET_MODELS.items():
    for alias in aliases:
        ALIAS_TO_MODEL[alias] = canonical
ALIASES_SORTED = sorted(ALIAS_TO_MODEL, key=len, reverse=True)

METRIC_COLS = [
    "accuracy", "precision", "recall", "f1", "shd", "acyclic",
    "skeleton_accuracy", "skeleton_precision", "skeleton_recall", "skeleton_f1",
    "ancestor_accuracy", "ancestor_precision", "ancestor_recall", "ancestor_f1",
    "nhd", "nhd_ratio",
    "prompt_tokens", "output_tokens", "total_tokens",
]

In [ ]:
def normalize_format(raw: str | None) -> str:
    if raw is None or raw == "":
        return "summary"  # historical default when no explicit format token appears
    raw = raw.lower()
    if raw in {"summary_joint", "joint", "payload"}:
        return "summary"
    if raw in {"summary", "matrix", "names_only"}:
        return raw
    if raw.startswith("matrix"):
        return "matrix"
    if raw.startswith("summary"):
        return "summary"
    if raw.startswith("cases"):
        return "cases"
    return raw


def infer_graph(path: Path) -> str | None:
    parts = path.parts
    for graph in GRAPHS:
        if graph in parts:
            return graph
    return None


def infer_model_from_name(name: str) -> tuple[str | None, str | None]:
    for alias in ALIASES_SORTED:
        if alias in name:
            return ALIAS_TO_MODEL[alias], alias
    # Baselines
    m = re.search(r"predictions_obs\d+_int\d+_([A-Za-z0-9_.-]+)(?:_seed\d+)?\.csv", name)
    if m:
        return m.group(1), m.group(1)
    return None, None


def parse_condition(path: Path) -> dict:
    name = path.name
    parent = path.parent.name
    graph = infer_graph(path)
    model, model_alias = infer_model_from_name(name)
    method_type = "llm"
    if model in {"PC", "GES", "ENCO"}:
        method_type = "data_only"

    # Strip metric suffixes to parse the underlying response filename.
    base = name
    for suffix in [".per_row.csv", ".summary.json", ".consensus_tau0.70.json", ".probplot.pdf", ".jsonl", ".csv"]:
        if base.endswith(suffix):
            base = base[: -len(suffix)]
            break

    # Names-only cells.
    names_only = base.startswith("responses_names_only")

    # Observational/interventional budgets.
    obs = None
    inter = None
    m = re.search(r"(?:responses|predictions)_obs(?P<obs>\d+)_int(?P<int>\d+)", base)
    if m:
        obs = int(m.group("obs"))
        inter = int(m.group("int"))
    elif names_only:
        obs = 0
        inter = 0

    # Semantic condition.
    semantic = None
    if names_only:
        semantic = "names_only"
    elif re.search(r"(^|_)anon(_|$)", base) or "_anon_" in parent or parent.startswith("cases_anon") or parent.startswith("matrix_anon"):
        semantic = "anon"
    elif re.search(r"(^|_)real(_|$)", base) or "_real_" in parent or parent.startswith("cases_real") or parent.startswith("matrix_real"):
        semantic = "real"
    else:
        semantic = "real"

    # Data format from filename first, then parent directory.
    fmt_raw = None
    if names_only:
        fmt_raw = "names_only"
    elif "summary_joint" in base:
        fmt_raw = "summary_joint"
    elif re.search(r"(^|_)summary(_|$)", base):
        fmt_raw = "summary"
    elif re.search(r"(^|_)matrix(_|$)", base):
        fmt_raw = "matrix"
    elif parent.startswith("summary"):
        fmt_raw = "summary"
    elif parent.startswith("matrix"):
        fmt_raw = "matrix"
    elif parent.startswith("cases"):
        fmt_raw = "cases"
    elif base.startswith("predictions_"):
        fmt_raw = "data_only"
    else:
        fmt_raw = "summary"

    # Prompt/run metadata.
    p = None
    m = re.search(r"_p(?P<p>\d+)(?:_|$)", base)
    if m:
        p = int(m.group("p"))
    shuf = None
    m = re.search(r"_shuf(?P<shuf>\d+)(?:_|$)", base)
    if m:
        shuf = int(m.group("shuf"))

    # Keep a rough variant string for provenance.
    variant = base
    if model_alias:
        variant = variant.replace(model_alias, "{MODEL}")
    variant = re.sub(r"responses_obs\d+_int\d+_", "responses_{BUDGET}_", variant)
    variant = re.sub(r"predictions_obs\d+_int\d+_", "predictions_{BUDGET}_", variant)
    variant = re.sub(r"responses_names_only_", "responses_names_only_", variant)

    return {
        "graph": graph,
        "method": model,
        "model_alias": model_alias,
        "method_type": method_type,
        "obs": obs,
        "inter": inter,
        "semantic": semantic,
        "format_raw": fmt_raw,
        "format": normalize_format(fmt_raw),
        "p": p,
        "shuffle": shuf,
        "variant": variant,
        "path": str(path.relative_to(REPO)),
    }


def summarize_per_row(path: Path) -> dict | None:
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        return {"read_error": str(exc)}
    out = {"source_kind": "per_row", "n_rows": len(df)}
    if "format_ok" in df.columns:
        out["valid_rows"] = int(pd.to_numeric(df["format_ok"], errors="coerce").fillna(0).sum())
    else:
        out["valid_rows"] = len(df)
    for col in METRIC_COLS:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            out[f"mean_{col}"] = float(vals.mean()) if vals.notna().any() else np.nan
            out[f"std_{col}"] = float(vals.std(ddof=1)) if vals.notna().sum() > 1 else np.nan
    return out


def summarize_json(path: Path) -> dict | None:
    try:
        data = json.loads(path.read_text())
    except Exception as exc:
        return {"read_error": str(exc)}
    out = {"source_kind": "summary_json"}
    mapping = {
        "num_rows": "n_rows",
        "valid_rows": "valid_rows",
        "avg_accuracy": "mean_accuracy",
        "avg_precision": "mean_precision",
        "avg_recall": "mean_recall",
        "avg_f1": "mean_f1",
        "avg_shd": "mean_shd",
        "consensus_f1": "consensus_f1",
        "consensus_shd": "consensus_shd",
        "brier": "brier",
        "brier_skeleton": "brier_skeleton",
        "var_f1_sd": "std_f1",
        "var_shd_sd": "std_shd",
    }
    for src, dst in mapping.items():
        if src in data:
            out[dst] = data[src]
    return out

In [ ]:
# Collect candidate result files.
records = []
for root in RESULT_ROOTS:
    if not root.exists():
        continue
    for graph in GRAPHS:
        graph_root = root / graph
        if not graph_root.exists():
            continue
        for path in graph_root.rglob("*"):
            if not path.is_file():
                continue
            name = path.name
            if not (
                name.endswith(".per_row.csv")
                or name.endswith(".summary.json")
            ):
                continue
            meta = parse_condition(path)
            if meta["graph"] not in GRAPHS:
                continue
            if meta["method"] is None:
                # Keep only target models and data-only baseline files by default.
                continue
            metrics = summarize_per_row(path) if name.endswith(".per_row.csv") else summarize_json(path)
            records.append({**meta, **(metrics or {})})

raw = pd.DataFrame(records)
print(f"Collected {len(raw)} metric artifacts")
print(raw.groupby(["graph", "source_kind"]).size().unstack(fill_value=0) if len(raw) else "No records")
raw.head()

In [ ]:
# Prefer per-row metrics over summary JSON for the same graph/method/condition/variant.
if raw.empty:
    agg = raw.copy()
else:
    raw["source_priority"] = raw["source_kind"].map({"per_row": 0, "summary_json": 1}).fillna(9)
    key_cols = [
        "graph", "method", "method_type", "obs", "inter", "semantic",
        "format", "format_raw", "p", "shuffle", "variant"
    ]
    agg = (
        raw.sort_values(["source_priority", "path"])
        .drop_duplicates(key_cols, keep="first")
        .drop(columns=["source_priority"])
        .reset_index(drop=True)
    )

# Paper-friendly condition labels.
def condition_label(row):
    if row.get("semantic") == "names_only":
        return "names_only"
    if row.get("method_type") == "data_only":
        return f"{row.get('method')} data-only"
    return f"{row.get('semantic')}+{row.get('format')}"

if not agg.empty:
    agg["condition"] = agg.apply(condition_label, axis=1)
    agg["budget"] = agg.apply(lambda r: f"N={int(r.obs) if pd.notna(r.obs) else 'NA'},M={int(r.inter) if pd.notna(r.inter) else 'NA'}", axis=1)
    agg["f1_shd"] = agg.apply(
        lambda r: f"{r.mean_f1:.3f} ({r.mean_shd:.1f})" if pd.notna(r.get("mean_f1")) and pd.notna(r.get("mean_shd")) else "",
        axis=1,
    )

agg.to_csv(OUT_DIR / "all_condition_metrics.csv", index=False)
print(f"Wrote {OUT_DIR / 'all_condition_metrics.csv'}")
agg.head(20)

In [ ]:
# Coverage table: which graph/model/condition/budget cells exist?
coverage_cols = ["graph", "method", "budget", "condition", "source_kind", "n_rows", "valid_rows", "path"]
coverage = agg[coverage_cols].sort_values(["graph", "method", "budget", "condition", "path"]) if not agg.empty else agg
coverage.to_csv(OUT_DIR / "coverage.csv", index=False)
coverage.head(100)

In [ ]:
# Main-paper candidate table.
# Adjust MAIN_BUDGET to whichever budget you choose for cross-graph reporting.
MAIN_BUDGET = (5000, 200)
MAIN_MODELS = [
    "gpt-5-mini",
    "gpt-5.2-pro",
    "Qwen/Qwen2.5-7B-Instruct-1M",
    "Qwen/Qwen2.5-14B-Instruct-1M",
    "Qwen/Qwen3-30B-A3B-Thinking-2507",
    "Qwen/Qwen2.5-72B-Instruct-AWQ",
    "meta-llama/Llama-3.1-70B-Instruct",
]

if agg.empty:
    main = agg.copy()
else:
    selected = agg[
        (agg["method"].isin(MAIN_MODELS + ["PC", "GES", "ENCO"]))
        & (
            (agg["semantic"].eq("names_only"))
            | ((agg["obs"].eq(MAIN_BUDGET[0])) & (agg["inter"].eq(MAIN_BUDGET[1])))
            | (agg["method_type"].eq("data_only"))
        )
    ].copy()

    # If there are multiple variants for the same paper cell, keep the best F1 by default,
    # but preserve the source path. For stricter reporting, filter `variant` before this step.
    paper_key = ["graph", "method", "budget", "condition"]
    main = (
        selected.sort_values(["mean_f1", "source_kind"], ascending=[False, True])
        .drop_duplicates(paper_key, keep="first")
        .sort_values(["graph", "method", "condition"])
    )

main.to_csv(OUT_DIR / "main_candidate_long.csv", index=False)
print(f"Wrote {OUT_DIR / 'main_candidate_long.csv'}")
main[["graph", "method", "budget", "condition", "mean_f1", "mean_shd", "n_rows", "valid_rows", "path"]].head(100)

In [ ]:
# Pivot for paper inspection: one row per graph/model/budget, one column per condition.
if main.empty:
    pivot = main.copy()
else:
    pivot = main.pivot_table(
        index=["graph", "method", "budget"],
        columns="condition",
        values="f1_shd",
        aggfunc="first",
    ).reset_index()

pivot.to_csv(OUT_DIR / "main_candidate_pivot.csv", index=False)
print(f"Wrote {OUT_DIR / 'main_candidate_pivot.csv'}")
pivot.head(100)

In [ ]:
# Compute matched contrast metrics where cells exist.
def get_score(df, graph, method, obs, inter, condition):
    sub = df[(df.graph == graph) & (df.method == method) & (df.condition == condition)]
    if obs is not None:
        sub = sub[sub.obs == obs]
    if inter is not None:
        sub = sub[sub.inter == inter]
    sub = sub.sort_values("mean_f1", ascending=False)
    if sub.empty:
        return np.nan, None
    row = sub.iloc[0]
    return row.mean_f1, row.path

contrast_rows = []
if not agg.empty:
    for graph in GRAPHS:
        for method in sorted(set(agg.method.dropna()) - {"PC", "GES", "ENCO"}):
            budgets = agg[(agg.graph == graph) & (agg.method == method)][["obs", "inter"]].dropna().drop_duplicates()
            for _, b in budgets.iterrows():
                obs, inter = int(b.obs), int(b.inter)
                name, _ = get_score(agg, graph, method, None, None, "names_only")
                real_sum, real_sum_path = get_score(agg, graph, method, obs, inter, "real+summary")
                anon_sum, anon_sum_path = get_score(agg, graph, method, obs, inter, "anon+summary")
                real_mat, real_mat_path = get_score(agg, graph, method, obs, inter, "real+matrix")
                anon_mat, anon_mat_path = get_score(agg, graph, method, obs, inter, "anon+matrix")
                for fmt, real, anon, rpath, apath in [
                    ("summary", real_sum, anon_sum, real_sum_path, anon_sum_path),
                    ("matrix", real_mat, anon_mat, real_mat_path, anon_mat_path),
                ]:
                    contrast_rows.append({
                        "graph": graph,
                        "method": method,
                        "obs": obs,
                        "inter": inter,
                        "format": fmt,
                        "mixed_information_gain_real": real - name if pd.notna(real) and pd.notna(name) else np.nan,
                        "mixed_information_gain_anon": anon - name if pd.notna(anon) and pd.notna(name) else np.nan,
                        "anonymization_drop": real - anon if pd.notna(real) and pd.notna(anon) else np.nan,
                        "real_path": rpath,
                        "anon_path": apath,
                    })
                contrast_rows.append({
                    "graph": graph,
                    "method": method,
                    "obs": obs,
                    "inter": inter,
                    "format": "summary_minus_matrix",
                    "data_format_gap_real": real_sum - real_mat if pd.notna(real_sum) and pd.notna(real_mat) else np.nan,
                    "data_format_gap_anon": anon_sum - anon_mat if pd.notna(anon_sum) and pd.notna(anon_mat) else np.nan,
                    "real_summary_path": real_sum_path,
                    "real_matrix_path": real_mat_path,
                    "anon_summary_path": anon_sum_path,
                    "anon_matrix_path": anon_mat_path,
                })

contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(OUT_DIR / "contrast_metrics.csv", index=False)
print(f"Wrote {OUT_DIR / 'contrast_metrics.csv'}")
contrasts.head(100)

In [ ]:
# Optional quick plots for exploration.
import matplotlib.pyplot as plt

plot_df = main.copy()
if not plot_df.empty:
    plot_df = plot_df[plot_df["mean_f1"].notna() & plot_df["condition"].isin(["names_only", "real+summary", "anon+summary", "real+matrix", "anon+matrix"])]
    for graph in GRAPHS:
        sub = plot_df[plot_df.graph == graph]
        if sub.empty:
            continue
        plt.figure(figsize=(10, 4))
        # Use compact model names in plot.
        labels = sub["method"].str.replace("Qwen/", "", regex=False).str.replace("meta-llama/", "", regex=False)
        x = np.arange(len(sub))
        plt.bar(x, sub["mean_f1"])
        plt.xticks(x, labels + "\n" + sub["condition"], rotation=75, ha="right", fontsize=8)
        plt.ylabel("Mean F1")
        plt.title(f"{graph}: candidate main-paper cells")
        plt.tight_layout()
        out = OUT_DIR / f"{graph}_candidate_f1.png"
        plt.savefig(out, dpi=200)
        plt.show()
        print(out)